# BEARS Colab Runner

Use this notebook with a GPU runtime. In Colab, go to `Runtime > Change runtime type` and select a GPU. With Colab Pro, choose the strongest available GPU only for long BDD runs; T4 is enough for smoke tests.

This notebook keeps your repo workflow local-first: clone your GitHub repo, mount Drive for large files such as `lastframe.zip`, install only Colab-safe dependencies, then run selected jobs through `colab_runner.py`.

In [ ]:
#@title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Clone or update your BEARS repo
# Put your modified repo here, not the upstream repo, because your local fixes are required.
REPO_URL = "https://github.com/YOUR_USERNAME/bears.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/bears"  #@param {type:"string"}

import os
from pathlib import Path

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repo that contains the Colab files and BEARS fixes.")

if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print("Repo ready at", os.getcwd())

In [ ]:
#@title 3. Install Colab-safe dependencies
%cd {REPO_DIR}
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.colab.txt

In [ ]:
#@title 4. Check GPU and imports
%cd {REPO_DIR}
!python colab_runner.py --job diagnostics

## Data locations

Recommended Drive layout:

```text
MyDrive/bears_data/lastframe.zip
MyDrive/bears_data/halfmnist-mnistdpl-dis-None-end.pt
```

`kand-3k.zip` is small enough to keep tracked in the repo. If your branch does not include it, upload it to `XOR_MNIST/data/kand-3k.zip` before running MiniKandinsky.

In [ ]:
#@title 5. Optional: copy HalfMNIST checkpoint from Drive
HALFMNIST_CKPT_IN_DRIVE = "/content/drive/MyDrive/bears_data/halfmnist-mnistdpl-dis-None-end.pt"  #@param {type:"string"}
from pathlib import Path

target = Path(REPO_DIR) / "XOR_MNIST" / "data" / "ckpts" / "halfmnist-mnistdpl-dis-None-end.pt"
source = Path(HALFMNIST_CKPT_IN_DRIVE)
target.parent.mkdir(parents=True, exist_ok=True)
if source.exists():
    !cp "{source}" "{target}"
    print("Copied", source, "to", target)
else:
    print("Checkpoint not found in Drive. Skipping copy:", source)

In [ ]:
#@title 6. Run a selected job
JOB = "halfmnist_smoke"  #@param ["xor_smoke", "halfmnist_smoke", "halfmnist_eval", "minikand_smoke", "bdd_preprocess_smoke", "bdd_train_smoke", "bdd_preprocess_full", "bdd_train_full", "archive_results"]
LASTFRAME_ZIP = "/content/drive/MyDrive/bears_data/lastframe.zip"  #@param {type:"string"}
BDD_OUTPUT = ""  #@param {type:"string"}
EPOCHS = 1  #@param {type:"integer"}
BATCH_SIZE = 64  #@param {type:"integer"}
BDD_BATCH_SIZE = 256  #@param {type:"integer"}
FEATURE_BATCH_SIZE = 64  #@param {type:"integer"}
FEATURE_WEIGHTS = "imagenet"  #@param ["imagenet", "none"]
LIMIT_PER_SPLIT = 8  #@param {type:"integer"}
EVAL_TYPE = "frequentist"  #@param ["frequentist", "mcdropout", "laplace", "bears", "deepensembles"]
USE_OOD = False  #@param {type:"boolean"}

args = [
    "python", "colab_runner.py",
    "--job", JOB,
    "--lastframe-zip", LASTFRAME_ZIP,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--bdd-batch-size", str(BDD_BATCH_SIZE),
    "--feature-batch-size", str(FEATURE_BATCH_SIZE),
    "--feature-weights", FEATURE_WEIGHTS,
    "--limit-per-split", str(LIMIT_PER_SPLIT),
    "--eval-type", EVAL_TYPE,
]
if BDD_OUTPUT:
    args += ["--bdd-output", BDD_OUTPUT]
if USE_OOD:
    args += ["--use-ood"]

import shlex
import subprocess

%cd {REPO_DIR}
print("Running:", " ".join(shlex.quote(arg) for arg in args))
completed = subprocess.run(args, check=False)
if completed.returncode:
    raise SystemExit(f"Job failed with exit code {completed.returncode}. Scroll above this message for the BEARS traceback.")

In [ ]:
#@title 7. Archive results for download or Drive copy
%cd {REPO_DIR}
!python colab_runner.py --job archive_results
!ls -lh colab_outputs

In [ ]:
#@title 8. Optional: copy latest result archive to Drive
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/bears_results"  #@param {type:"string"}
!mkdir -p "{DRIVE_RESULTS_DIR}"
!cp -v $(ls -t colab_outputs/bears_results_*.zip | head -1) "{DRIVE_RESULTS_DIR}/"